# Kaggle GPU OCR Backend Server (12-Hour Headless Mode)

This notebook installs PaddleOCR GPU version and runs a remote GPU API server for up to **12 hours straight** in the cloud.

---

### **How to Run for 12 Hours Straight Without Keeping Your Browser Open:**

1. **GPU Accelerator (Mandatory):**
   - On the right sidebar, expand **Notebook Options** -> **Accelerator**.
   - Select **GPU T4 x2** (or GPU P100).
2. **Turn Internet ON (Mandatory):**
   - Under **Notebook Options** -> **Internet**, toggle to **ON**.
3. **Headless Channel Name (Optional):**
   - In Cell 3, you can customize `NOTIFY_CHANNEL = "onebhoomi_ocr_tunnel"` (e.g. `"onebhoomi_ocr_mykey"`).
4. **Start Headless Run (Crucial Step):**
   - At the top right of the Kaggle interface, click **Save Version**.
   - Under *Version Type*, choose **Save & Run All (Commit)**.
   - Click **Save**.
5. **Close Browser:**
   - **You can now immediately close your browser, turn off Wi-Fi, or shut down your computer!**
   - Kaggle executes in an isolated GPU container for up to 12 hours.
6. **Sync URL to Your Local Web App:**
   - On your local computer, simply run:
     ```bash
     ./update_ocr_url.sh --sync
     ```
   - It will automatically fetch the live Kaggle GPU URL and hook it to your local registry console!
   - You can also check your live URL on your phone or browser at `https://ntfy.sh/onebhoomi_ocr_tunnel`.


In [ ]:
# Cell 1: Environment Warmup
import sys
print("✓ Kaggle session started successfully!")
print(f"Python Version: {sys.version}")

In [ ]:
# Cell 2: Uninstall conflicting libraries & install PaddlePaddle GPU + PaddleOCR
# 1. Uninstall pre-installed torch to avoid CUDA driver conflicts with Paddle
!pip uninstall -y torch torchvision

# 2. Detect local CUDA version and install matching PaddlePaddle GPU package
import re
import subprocess

def detect_cuda_version():
    try:
        nvcc_out = subprocess.check_output(["nvcc", "--version"]).decode("utf-8")
        match = re.search(r"release (\d+\.\d+)", nvcc_out)
        if match:
            return match.group(1)
    except Exception:
        pass
    try:
        smi_out = subprocess.check_output(["nvidia-smi"]).decode("utf-8")
        match = re.search(r"CUDA Version:\s*(\d+\.\d+)", smi_out)
        if match:
            return match.group(1)
    except Exception:
        pass
    return None

cuda_version = detect_cuda_version()
print(f"Detected CUDA version: {cuda_version}")

cu_suffix = "cu120"
if cuda_version:
    try:
        parts = cuda_version.split(".")
        major = int(parts[0])
        minor = int(parts[1]) if len(parts) > 1 else 0
        if major == 11:
            cu_suffix = "cu118"
        elif major == 12:
            if minor >= 6:
                cu_suffix = "cu126"
            elif minor >= 2:
                cu_suffix = "cu122"
            else:
                cu_suffix = "cu120"
        else:
            cu_suffix = f"cu{major}{minor}"
    except Exception:
        pass

print(f"Installing PaddlePaddle-GPU for {cu_suffix}...")
# Added --default-timeout=1000 to prevent timeout on large downloads
!pip install --default-timeout=1000 paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/{cu_suffix}/

print("\nInstalling PaddleOCR 3.7.0...")
!pip install paddleocr==3.7.0

In [ ]:
# Cell 3: Start GPU OCR Engine, Expose Public Tunnel & Run 12-Hour Headless Supervisor
import os
import re
import sys
import time
import cv2
import shutil
import platform
import tempfile
import requests
import subprocess
import threading
import numpy as np
from pathlib import Path
from flask import Flask, request, jsonify
from paddleocr import PaddleOCR

# ==============================================================================
# 1. HEADLESS NOTIFICATION CONFIGURATION
# ==============================================================================
# When running in background ("Save & Run All"), Kaggle runs headlessly without a browser.
# The server will broadcast its live public URL to this channel so your local web app
# can automatically pull it via `./update_ocr_url.sh --sync`.
NOTIFY_CHANNEL = "onebhoomi_ocr_tunnel"

# Maximum duration to keep running (Kaggle limit is 12 hours = 43200s; we run for 11.5 hours)
MAX_RUNTIME_HOURS = 11.5

def broadcast_url(url, note="Server Online"):
    if not NOTIFY_CHANNEL:
        return
    endpoint = f"https://ntfy.sh/{NOTIFY_CHANNEL}"
    try:
        requests.post(
            endpoint,
            data=url.encode("utf-8"),
            headers={
                "Title": f"Kaggle GPU OCR ({note})",
                "Tags": "rocket,gpu,computer",
                "Priority": "high"
            },
            timeout=10
        )
        print(f"✓ Broadcasted live URL to {endpoint}")
    except Exception as exc:
        print(f"Notification broadcast note: {exc}")

# ==============================================================================
# 2. TUNNEL MANAGEMENT (Cloudflared & Localtunnel)
# ==============================================================================
_CLOUDFLARED_BINARY_URLS = {
    "x86_64": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "amd64": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "aarch64": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",
    "arm64": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",
}

active_tunnel_proc = None

def _stream_until_url(proc, patterns, timeout_s=120):
    global active_tunnel_proc
    active_tunnel_proc = proc
    if proc.stdout is None:
        raise RuntimeError("Tunnel process stdout is not available.")
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        line = proc.stdout.readline()
        if line:
            print(line, end="")
            for pattern in patterns:
                match = pattern.search(line)
                if match:
                    return match.group(0)
        elif proc.poll() is not None:
            break
        else:
            time.sleep(0.2)
    raise TimeoutError("Timed out while waiting for a public tunnel URL.")

def _start_localtunnel(port):
    lt_cmd = None
    npx_cmd = shutil.which("npx.cmd") or shutil.which("npx")
    lt_cmd_bin = shutil.which("lt.cmd") or shutil.which("lt")
    if npx_cmd:
        lt_cmd = [npx_cmd, "-y", "localtunnel", "--port", str(port)]
    elif lt_cmd_bin:
        lt_cmd = [lt_cmd_bin, "--port", str(port)]
    if not lt_cmd:
        raise FileNotFoundError("Neither 'npx' nor 'lt' is available.")
    print("Starting localtunnel...")
    proc = subprocess.Popen(lt_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    return _stream_until_url(proc, [re.compile(r"https://[^\s]+(?:localtunnel\.me|loca\.lt)[^\s]*", re.IGNORECASE)])

def _download_cloudflared_binary():
    machine = platform.machine().lower()
    asset_url = _CLOUDFLARED_BINARY_URLS.get(machine)
    if not asset_url:
        raise RuntimeError(f"Unsupported architecture for cloudflared: {machine}")
    target_dir = Path(tempfile.gettempdir()) / "codex-cloudflared"
    target_dir.mkdir(parents=True, exist_ok=True)
    binary_path = target_dir / "cloudflared"
    if binary_path.exists():
        return str(binary_path)
    print(f"Downloading cloudflared from {asset_url}...")
    response = requests.get(asset_url, timeout=120)
    response.raise_for_status()
    binary_path.write_bytes(response.content)
    binary_path.chmod(0o755)
    return str(binary_path)

def _start_cloudflared(port):
    binary = shutil.which("cloudflared") or _download_cloudflared_binary()
    print("Starting cloudflared quick tunnel...")
    proc = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    return _stream_until_url(proc, [re.compile(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", re.IGNORECASE)])

def expose_port(port, prefer_kaggle=None):
    kaggle_env = any(k in os.environ for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "KAGGLE_WORKING_DIR"))
    prefer_kaggle = kaggle_env if prefer_kaggle is None else prefer_kaggle
    attempts = [_start_cloudflared, _start_localtunnel] if prefer_kaggle else [_start_localtunnel, _start_cloudflared]
    errors = []
    for starter in attempts:
        try:
            url = starter(port)
            print(f"\nPublic OCR URL: {url}")
            return url
        except Exception as exc:
            errors.append(f"{starter.__name__}: {exc}")
            print(f"{starter.__name__} failed: {exc}")
    raise RuntimeError("Unable to create a public tunnel: " + " | ".join(errors))

# ==============================================================================
# 3. INITIALIZE PADDLEOCR ENGINE (GPU ACCELERATED)
# ==============================================================================
print("Initializing PaddleOCR engine with GPU...")
ocr = PaddleOCR(
    det_limit_side_len=1536,
    device='gpu',
    lang='en',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)
ocr_lock = threading.Lock()
print("PaddleOCR GPU engine initialized successfully!")

# ==============================================================================
# 4. FLASK OCR API SERVER
# ==============================================================================
gpu_name = "NVIDIA GPU"
try:
    smi_out = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode("utf-8")
    gpu_name = smi_out.strip()
except Exception:
    pass
print(f"Detected GPU: {gpu_name}")

app = Flask(__name__)

@app.route("/", methods=["GET"])
@app.route("/status", methods=["GET"])
def status_endpoint():
    return jsonify({"status": "connected", "gpu_name": gpu_name})

@app.route("/ocr", methods=["POST"])
def ocr_endpoint():
    if "image" not in request.files:
        return jsonify({"error": "No image file provided"}), 400
    file = request.files["image"]
    img_bytes = file.read()
    nparr = np.frombuffer(img_bytes, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if image is None:
        return jsonify({"error": "Failed to decode image"}), 400
    with ocr_lock:
        start_time = time.perf_counter()
        result = ocr.predict(image)[0]
        ocr_time = (time.perf_counter() - start_time) * 1000
    rec_polys_list = [poly.tolist() for poly in result["rec_polys"]]
    return jsonify({
        "rec_texts": result["rec_texts"],
        "rec_scores": [float(s) for s in result["rec_scores"]],
        "rec_polys": rec_polys_list,
        "ocr_time_ms": ocr_time,
        "gpu_name": gpu_name
    })

def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()
time.sleep(2)
print("Flask server successfully started on port 5000!")

print("\n=== Exposing Kaggle Port 5000 ===")
public_url = expose_port(5000)

# Broadcast immediately to ntfy.sh
broadcast_url(public_url, note="Server Ready")

# ==============================================================================
# 5. SUPERVISED 12-HOUR HEADLESS KEEP-ALIVE SUPERVISOR
# ==============================================================================
deadline = time.time() + (MAX_RUNTIME_HOURS * 3600)

print("\n" + "=" * 65)
print(" 🚀 KAGGLE 12-HOUR HEADLESS OCR BACKEND IS RUNNING")
print("=" * 65)
print(f" • Public Tunnel URL: {public_url}")
print(f" • Sync Channel:     https://ntfy.sh/{NOTIFY_CHANNEL}")
print(f" • Planned Runtime:  {MAX_RUNTIME_HOURS} hours")
print(" • You can close your browser tab or turn off your PC safely!")
print(f" • Local sync:       ./update_ocr_url.sh --sync")
print("=" * 65 + "\n")

loop_count = 0
try:
    while time.time() < deadline:
        time.sleep(60)
        loop_count += 1

        # Check if tunnel process is still alive; restart if crashed
        if active_tunnel_proc and active_tunnel_proc.poll() is not None:
            print("⚠️ Tunnel dropped! Re-establishing tunnel...")
            try:
                public_url = expose_port(5000)
                broadcast_url(public_url, note="Tunnel Reconnected")
            except Exception as e:
                print(f"Tunnel restart attempt failed: {e}")

        # Heartbeat log every 10 minutes (prevents Kaggle log stagnation)
        if loop_count % 10 == 0:
            elapsed_m = loop_count
            remain_m = max(0, int((deadline - time.time()) / 60))
            try:
                st = requests.get("http://127.0.0.1:5000/status", timeout=5).json()
                st_desc = f"OK ({st.get('gpu_name', 'GPU Online')})"
            except Exception as ex:
                st_desc = f"Warning: {ex}"
            print(f"[{time.strftime('%H:%M:%S')}] Heartbeat: Active {elapsed_m}m / {remain_m}m remaining | Local: {st_desc}")

        # Re-broadcast URL every 60 minutes
        if loop_count % 60 == 0:
            broadcast_url(public_url, note=f"Active ({loop_count // 60}h)")

    print("Target 11.5-hour execution completed successfully.")
except KeyboardInterrupt:
    print("Execution manually interrupted.")
